# 05. Риски: моделирование Монте-Карло и дерево решений

**ЛР 4, шаг 2.** Ноутбук отвечает на вопрос спонсора «с какой вероятностью мы успеем» с учётом неопределённости оценок и рисков. Он много раз «проигрывает» проект со случайными длительностями работ и случайным наступлением рисков, а затем считает, в какой доле прогонов проект укладывается в срок и бюджет.

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OWNER/REPO/blob/main/notebooks/05_risk_monte_carlo.ipynb)

Замените в ссылке `OWNER/REPO` на свой репозиторий. Данные и Google Диск не нужны. Запуск: **Среда выполнения → Выполнить все**.

По умолчанию таблицы заполнены примером AquaMap: работы и оценки — из ЛР 3, риски — из реестра ЛР 4. Замените их своими. Ноутбук считает вероятности, но решения по рискам принимает команда.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import date, timedelta
pd.set_option("display.float_format", lambda x: f"{x:,.2f}".replace(",", " "))
rng = np.random.default_rng(42)   # фиксированное зерно: у всех получатся одинаковые цифры

## 1. Исходные данные

**Работы** — те же, что в ноутбуке 04 (ЛР 3): оценки O, M, P в часах, длительность в рабочих днях по базовому плану, предшественники.

**Риски** — события из реестра, которые можно выразить в днях и часах:
- `task` — какая работа удлиняется, если риск наступил;
- `p_before` — вероятность наступления за проект (0–1) до ответных мер;
- `days_before` — на сколько рабочих дней удлиняется работа;
- `hours_before` — сколько дополнительных часов работы это потребует;
- `p_after`, `days_after`, `hours_after` — то же после ответных мер из реестра рисков.

Ответная мера может снижать вероятность (выборочная проверка данных, резервный источник) или последствия (дублёр по задаче). Сравнение сценариев показывает, что даёт план реагирования.

**Цели:** срок по базовому плану, срок с буфером, плановая трудоёмкость и трудоёмкость с резервом на риски.

In [ ]:
TASKS = [
    # code, wbs,  name,                                     O,  M,  P, days, after
    ("A", "5.1", "Окружение и автопроверки",                 4,  6, 12,  5, ""),
    ("B", "1.5", "Проверка полноты и уточнение застройки",   4,  8, 16,  9, ""),
    ("C", "2.1", "Показатели по районам и школы",            5,  8, 14,  9, ""),
    ("F", "3.1", "Веб-карта: слой точек",                    6, 10, 20, 12, "A"),
    ("D", "2.2", "Доступность 500 м",                        6,  9, 18, 10, "B"),
    ("E", "2.3", "Парки без воды",                           3,  5,  9,  5, "B"),
    ("G", "3.2", "Картограмма районов",                      4,  6, 12,  5, "F C D"),
    ("H", "4.1", "Рекомендации по размещению",               6,  8, 14,  6, "C D E"),
    ("I", "5.2", "Публикация на GitHub Pages",               3,  5, 12,  5, "G"),
    ("J", "4.2", "Выгрузки для департамента",                2,  3,  6,  3, "H"),
    ("K", "6.3", "Итоговый отчёт и защита",                  8, 12, 20,  5, "I J"),
]

RISKS = [
    # id,  описание,                                     task, p_before, days, hours, p_after, days_after, hours_after
    ("R1", "Неполнота OSM: повторная проверка",              "B",  0.50,    3,    6,     0.20,    3,          6),
    ("R2", "Участник выпадает на неделю",                    "D",  0.20,    5,    0,     0.20,    2,          2),
    ("R3", "Заказчик меняет требования к рекомендациям",     "H",  0.30,    2,    6,     0.30,    2,          6),
    ("R6", "Overpass недоступен при финальном обновлении",   "I",  0.40,    2,   12,     0.05,    2,         12),
]

START = date(2026, 10, 26)
HOLIDAYS = {date(2026, 11, 4)}
DEADLINE = date(2026, 12, 11)        # конец работ по базовому плану
BUFFER_END = date(2026, 12, 17)      # последний день буфера проекта
PLAN_HOURS = 87.3                    # ожидаемая трудоёмкость E из ЛР 3
RESERVE_HOURS = 15                   # резерв на риски из бюджета ЛР 3
N = 10_000                           # число прогонов

tasks = pd.DataFrame(TASKS, columns=["code", "wbs", "name", "O", "M", "P", "days", "after"]).set_index("code")
tasks["after"] = tasks["after"].str.split()
tasks["E"] = (tasks["O"] + 4 * tasks["M"] + tasks["P"]) / 6
risks = pd.DataFrame(RISKS, columns=["id", "name", "task", "p_before", "days_before", "hours_before",
                                    "p_after", "days_after", "hours_after"]).set_index("id")
risks["p_none"], risks["days_none"], risks["hours_none"] = 0.0, 0, 0
assert set(risks["task"]) <= set(tasks.index), "В рисках указана несуществующая работа"
risks

## 2. Календарь и порядок работ

Рабочие дни переводятся в даты с учётом выходных и праздников, как в ноутбуке 04. Работы упорядочиваются так, чтобы предшественник всегда шёл раньше последователя.

In [ ]:
def workdays(start, n):
    days, d = [], start
    while len(days) < n:
        if d.weekday() < 5 and d not in HOLIDAYS:
            days.append(d)
        d += timedelta(days=1)
    return days

CAL = workdays(START, 200)
def day_to_date(x):
    # x — число рабочих дней от старта (дробное), возвращает дату последнего рабочего дня
    return CAL[max(int(np.ceil(x)) - 1, 0)]
def date_to_day(d):
    return CAL.index(d) + 1

order, seen = [], set()
def visit(k):
    if k in seen: return
    for p in tasks.at[k, "after"]: visit(p)
    seen.add(k); order.append(k)
for k in tasks.index: visit(k)

DEADLINE_DAY, BUFFER_DAY = date_to_day(DEADLINE), date_to_day(BUFFER_END)
print("Порядок работ:", " → ".join(order))
print(f"Срок по плану: день {DEADLINE_DAY} ({DEADLINE:%d.%m}), конец буфера: день {BUFFER_DAY} ({BUFFER_END:%d.%m})")

## 3. Модель

В каждом прогоне:
1. **Трудоёмкость** каждой работы выбирается случайно из бета-PERT распределения по O, M, P. Это то же распределение, на котором основана формула \(E = (O + 4M + P) / 6\).
2. **Длительность** меняется пропорционально трудоёмкости: если работа потребовала на 20% больше часов, чем E, она длится на 20% дольше. Это упрощение: в жизни часть задержки можно поглотить сверхурочно.
3. **Каждый риск** наступает с вероятностью из выбранного сценария и добавляет дни и часы к своей работе.
4. **Срок проекта** считается прямым проходом по сетевому графику, как в CPM.

Результат — 10 000 возможных исходов проекта, по которым считаются вероятности.

In [ ]:
def beta_pert(o, m, p, size):
    if p == o:
        return np.full(size, float(m))
    a = 1 + 4 * (m - o) / (p - o)
    b = 1 + 4 * (p - m) / (p - o)
    return o + rng.beta(a, b, size) * (p - o)

def simulate(scenario):
    hours = pd.DataFrame({k: beta_pert(*tasks.loc[k, ["O", "M", "P"]], N) for k in tasks.index})
    dur = hours.div(tasks["E"]) * tasks["days"]
    fired = pd.DataFrame({r: rng.random(N) < risks.at[r, f"p_{scenario}"] for r in risks.index})
    for r in risks.index:
        t = risks.at[r, "task"]
        dur[t] += fired[r] * risks.at[r, f"days_{scenario}"]
        hours[t] += fired[r] * risks.at[r, f"hours_{scenario}"]
    ef = pd.DataFrame(index=dur.index)
    for k in order:
        preds = tasks.at[k, "after"]
        es = ef[preds].max(axis=1) if preds else 0
        ef[k] = es + dur[k]
    finish = ef.max(axis=1)
    # критический путь в каждом прогоне: обратный проход (LF = минимум LS последователей)
    ls = pd.DataFrame(index=dur.index)
    succ = {k: [s for s in tasks.index if k in tasks.at[s, "after"]] for k in tasks.index}
    for k in reversed(order):
        lf = ls[succ[k]].min(axis=1) if succ[k] else finish
        ls[k] = lf - dur[k]
    slack = ls[order] - (ef[order] - dur[order])
    critical = slack.abs() < 1e-6
    return dict(hours=hours, dur=dur, fired=fired, finish=finish, total_hours=hours.sum(axis=1), critical=critical)

only_estimates = simulate("none")
before = simulate("before")
after = simulate("after")
print("Готово:", N, "прогонов для каждого сценария")

## 4. Срок и трудоёмкость: вероятности

- **P(в срок)** — доля прогонов, где проект закончен не позже срока базового плана.
- **P(в буфер)** — не позже последнего дня буфера.
- **P50 и P85** — срок или трудоёмкость, которые не будут превышены с вероятностью 50% и 85%. В отчёте спонсору обычно называют P85: это обещание, которое команда выполнит в большинстве случаев.

In [ ]:
def summary(res, title):
    f, h = res["finish"], res["total_hours"]
    p50, p85 = np.percentile(f, [50, 85])
    return {
        "Сценарий": title,
        "P(в срок), %": 100 * (f <= DEADLINE_DAY).mean(),
        "P(в буфер), %": 100 * (f <= BUFFER_DAY).mean(),
        "Срок P50": day_to_date(p50).strftime("%d.%m"),
        "Срок P85": day_to_date(p85).strftime("%d.%m"),
        "Часы P50": np.percentile(h, 50),
        "Часы P85": np.percentile(h, 85),
        f"P(часы ≤ {PLAN_HOURS:g}), %": 100 * (h <= PLAN_HOURS).mean(),
        f"P(часы ≤ {PLAN_HOURS + RESERVE_HOURS:g}), %": 100 * (h <= PLAN_HOURS + RESERVE_HOURS).mean(),
    }

result = pd.DataFrame([summary(only_estimates, "Только неопределённость оценок"),
                       summary(before, "Оценки и риски до ответных мер"),
                       summary(after, "Оценки и риски после ответных мер")]).set_index("Сценарий")
result.T

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
bins = np.arange(int(min(before["finish"].min(), after["finish"].min())), int(before["finish"].max()) + 2)
for res, label in [(before, "до ответных мер"), (after, "после ответных мер")]:
    axes[0].hist(res["finish"], bins=bins, alpha=0.5, label=label)
    axes[1].hist(res["total_hours"], bins=40, alpha=0.5, label=label)
axes[0].axvline(DEADLINE_DAY, color="black", ls="--", label=f"срок {DEADLINE:%d.%m}")
axes[0].axvline(BUFFER_DAY, color="red", ls="--", label=f"конец буфера {BUFFER_END:%d.%m}")
axes[0].set_xlabel("Окончание, рабочий день от старта"); axes[0].set_ylabel("Число прогонов"); axes[0].legend()
axes[1].axvline(PLAN_HOURS, color="black", ls="--", label=f"план {PLAN_HOURS:g} ч")
axes[1].axvline(PLAN_HOURS + RESERVE_HOURS, color="red", ls="--", label="план + резерв")
axes[1].set_xlabel("Трудоёмкость, ч"); axes[1].legend()
fig.suptitle("Монте-Карло: срок и трудоёмкость проекта")
fig.tight_layout()
fig.savefig("monte_carlo.png", dpi=150)
plt.show()

## 5. Какие работы и риски сильнее всего влияют на срок

- **Индекс критичности** — в какой доле прогонов работа оказалась на критическом пути. Работа с индексом 30% не критическая в базовом плане, но в каждом третьем исходе определяет срок.
- **Чувствительность** — корреляция длительности работы со сроком проекта (ранговая, Спирмена). Чем выше, тем сильнее задержка этой работы сдвигает финиш.
- **Влияние риска** — на сколько дней в среднем позже заканчивается проект в прогонах, где риск наступил, по сравнению с прогонами, где нет.

In [ ]:
res = before
crit = pd.DataFrame({
    "wbs": tasks["wbs"], "name": tasks["name"],
    "Индекс критичности, %": 100 * res["critical"].mean(),
    "Чувствительность": [res["dur"][k].rank().corr(res["finish"].rank()) for k in tasks.index],
}).sort_values("Чувствительность", ascending=False)
crit

In [ ]:
impact = pd.DataFrame({
    "Риск": risks["name"],
    "Наступил в прогонах, %": 100 * res["fired"].mean(),
    "Сдвиг финиша, дн": [res["finish"][res["fired"][r]].mean() - res["finish"][~res["fired"][r]].mean() for r in risks.index],
    "P(в срок) без риска, %": [100 * (res["finish"][~res["fired"][r]] <= DEADLINE_DAY).mean() for r in risks.index],
})
impact.sort_values("Сдвиг финиша, дн", ascending=False)

## 6. Дерево решений для одного риска

Для риска выбирают одну из альтернатив. У каждой альтернативы есть затраты сразу и возможные исходы с вероятностями. **Ожидаемая денежная стоимость (EMV)** — сумма «затраты + вероятность × последствия». Здесь всё считается в часах и днях задержки, потому что ставка условная.

Пример AquaMap для риска R6:
- **A. Рассчитывать на Overpass.** Ничего не делаем заранее. С вероятностью 40% Overpass недоступен при финальном обновлении: 12 часов на ручную выгрузку и повторный прогон ноутбуков, 2 дня задержки на критическом пути.
- **B. Сделать резервную загрузку с Geofabrik.** 4 часа сразу на доработку ноутбука 01. Остаточный риск 5%: ошибка в резервном варианте, те же последствия.

In [ ]:
OPTIONS = [
    # вариант,                      затраты сейчас, ч; исходы: (вероятность, часы, дни задержки)
    ("A. Рассчитывать на Overpass",  0, [(0.40, 12, 2), (0.60, 0, 0)]),
    ("B. Резерв через Geofabrik",    4, [(0.05, 12, 2), (0.95, 0, 0)]),
]
rows = []
for name, cost, outcomes in OPTIONS:
    assert abs(sum(p for p, _, _ in outcomes) - 1) < 1e-9, f"{name}: вероятности не дают в сумме 1"
    rows.append({
        "Вариант": name,
        "Затраты сейчас, ч": cost,
        "Ожидаемые часы (EMV), ч": cost + sum(p * h for p, h, _ in outcomes),
        "Ожидаемая задержка, дн": sum(p * d for p, _, d in outcomes),
        "Худший случай, ч": cost + max(h for _, h, _ in outcomes),
    })
pd.DataFrame(rows).set_index("Вариант")

## 7. Что перенести в отчёт

В `docs/risk-register.md`, раздел «Количественный анализ», перенесите:
1. Таблицу из раздела 4 (все три сценария) и график `monte_carlo.png` (скачайте его ячейкой ниже и положите в `docs/img/`).
2. Три работы с наибольшей чувствительностью и риски с наибольшим сдвигом финиша.
3. Таблицу дерева решений и решение команды с обоснованием.

Сформулируйте вывод для спонсора одной фразой: «С вероятностью X% проект завершится до …, с вероятностью Y% — до конца буфера. Главные угрозы сроку: …».

In [ ]:
try:
    from google.colab import files
    files.download("monte_carlo.png")
except ImportError:
    print("Локальный запуск: файл monte_carlo.png сохранён рядом с ноутбуком")